# Fragment Track — Ink Detection (PHercParis4 Scroll Decipherment)

**Purpose:** Use the Kaggle competition fragments — the only data in this project with *manually confirmed* ink labels (annotated from IR photography) — to train a 3D→2D CNN, then apply it to the actual PHercParis4 scroll segments to generate ink probability maps. These maps feed into stage 2 of the decipherment pipeline (letter isolation → LLM gap-fill → transcription).

**Why fragments first?** Fragment labels are ground truth. The segment track (`segment_ink_detection.ipynb`) uses pseudo-labels from a prior model, which are noisier. A fragment-trained model gives a clean baseline before introducing pseudo-label noise.

**Pipeline position:** Stage 1 (ink detection) → outputs `predictions/{segment_id}_prob.png` and `_ink.png`

**Data layout:**
```
data/
  train/                         ← Kaggle fragments (manual ink labels)
    fragment1/
      surface_volume/            ← 00.tif … 64.tif (65 slices, CT Z-stack)
      mask.png                   ← valid papyrus region
      inklabels.png              ← ground-truth binary ink mask (IR-confirmed)
    fragment2/ fragment3/ ...
  scroll/                        ← PHercParis4 segments for inference (~15 GB, 7 segments)
    20230827161847/              ← Grand Prize segment (first confirmed readable text)
      surface_volume/            ← 00.tif … 64.tif
      mask.png
    20230826170124/ ... (6 more)
```

> **No ink labels exist for scroll segments.** `mask.png` marks valid papyrus pixels only.
> Run `python download_segments.py` to fetch scroll data before section 9.

## 0 — Install & imports

In [ ]:
# Run once — install from requirements.txt
# !pip install -r requirements.txt --index-url https://download.pytorch.org/whl/cu118

In [ ]:
import gc
import random
from pathlib import Path

import cv2
import numpy as np
import tifffile
import imageio
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import albumentations as A

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

## 1 — Config

In [ ]:
CFG = dict(
    # Paths
    train_dir   = Path('../data/train'),
    scroll_dir  = Path('../data/scroll'),   # PHercParis4 segments (7 downloaded)
    model_dir   = Path('../models'),
    pred_dir    = Path('../predictions'),

    # Surface volume — all 65 layers downloaded
    z_start     = 0,
    z_end       = 65,

    # Patch extraction
    patch_size  = 224,
    stride      = 112,   # 50% overlap

    # Training
    batch_size  = 8,
    num_epochs  = 15,
    lr          = 1e-4,
    val_fragment = 'fragment1',
    seed        = 42,

    # Inference
    threshold   = 0.45,
    min_cc_size = 50,

    # Scroll segments selected for download (~15.08 GB total, all 65 layers)
    scroll_segments = [
        '20230827161847',   # 5.73 GB — Grand Prize, first readable text
        '20230826170124',   # 2.35 GB — nearby region
        '20230820174948',   # 1.87 GB — nearby region
        '20230826135043',   # 1.55 GB — nearby region
        '20230828154913',   # 1.40 GB — day after Grand Prize
        '20230819093803',   # 1.15 GB — nearby region
        '20230504093154',   # 1.03 GB — smaller early segment
    ],
)

CFG['model_dir'].mkdir(exist_ok=True)
CFG['pred_dir'].mkdir(exist_ok=True)

random.seed(CFG['seed'])
np.random.seed(CFG['seed'])
torch.manual_seed(CFG['seed'])

## 2 — Data loading helpers

In [ ]:
def load_surface_volume(seg_dir: Path, z_start=0, z_end=65) -> np.ndarray:
    """Load TIFF stack → float32 (Z, H, W) normalised to [0, 1]."""
    paths = sorted((seg_dir / 'surface_volume').glob('*.tif'))[z_start:z_end]
    volume = np.stack([tifffile.imread(str(p)).astype(np.float32) for p in paths])
    vmin, vmax = volume.min(), volume.max()
    if vmax > vmin:
        volume = (volume - vmin) / (vmax - vmin)
    return volume


def load_mask(seg_dir: Path) -> np.ndarray:
    """Binary papyrus mask (H, W) — works for fragments and scroll segments."""
    # Fragments use mask.png; scroll segments store it as {id}_mask.png
    # download_segments.py normalises the name to mask.png so one path works.
    p = seg_dir / 'mask.png'
    if not p.exists():
        return None  # caller should handle missing mask
    return (np.array(Image.open(p).convert('L')) > 127).astype(np.uint8)


def load_labels(frag_dir: Path) -> np.ndarray:
    """Binary ink label (H, W) — only exists for training fragments."""
    lbl = np.array(Image.open(frag_dir / 'inklabels.png').convert('L'))
    return (lbl > 127).astype(np.uint8)


def extract_patches(volume, mask, labels=None, patch_size=224, stride=112):
    """Sliding-window patch extraction. Returns list of (vol_crop, lbl_crop, (y, x))."""
    _, H, W = volume.shape
    patches = []
    for y in range(0, H - patch_size + 1, stride):
        for x in range(0, W - patch_size + 1, stride):
            if mask[y:y+patch_size, x:x+patch_size].mean() < 0.25:
                continue
            vol_crop = volume[:, y:y+patch_size, x:x+patch_size]
            lbl_crop = labels[y:y+patch_size, x:x+patch_size] if labels is not None else None
            patches.append((vol_crop, lbl_crop, (y, x)))
    return patches

## 3 — Dataset & augmentation

In [ ]:
TRAIN_AUG = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15, rotate_limit=30, p=0.5),
    A.OneOf([
        A.GaussNoise(var_limit=(0.0, 0.01), p=1.0),
        A.GaussianBlur(blur_limit=3, p=1.0),
    ], p=0.3),
    A.CoarseDropout(max_holes=8, max_height=16, max_width=16, fill_value=0, p=0.3),
])


class VesuviusDataset(Dataset):
    def __init__(self, patches, augment=False):
        self.patches = patches
        self.augment = augment

    def __len__(self):
        return len(self.patches)

    def __getitem__(self, idx):
        vol, lbl, _ = self.patches[idx]
        if self.augment:
            aug = TRAIN_AUG(image=vol.transpose(1, 2, 0), mask=lbl.astype(np.float32))
            vol = aug['image'].transpose(2, 0, 1)
            lbl = aug['mask']
        return torch.from_numpy(vol).float(), torch.from_numpy(lbl).float().unsqueeze(0)

## 4 — Model (3D encoder + 2D decoder)

In [ ]:
class ConvBnRelu3d(nn.Sequential):
    def __init__(self, in_c, out_c, k=3, s=1, p=1):
        super().__init__(
            nn.Conv3d(in_c, out_c, k, stride=s, padding=p, bias=False),
            nn.BatchNorm3d(out_c),
            nn.ReLU(inplace=True),
        )

class ConvBnRelu2d(nn.Sequential):
    def __init__(self, in_c, out_c, k=3, s=1, p=1):
        super().__init__(
            nn.Conv2d(in_c, out_c, k, stride=s, padding=p, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
        )


class VesuviusNet(nn.Module):
    """
    3D encoder collapses the Z (depth) dimension, 2D UNet decoder produces the ink mask.
    Input : (B, Z, H, W)   Output: (B, 1, H, W) logits
    """
    def __init__(self):
        super().__init__()
        self.enc3d_1 = nn.Sequential(
            ConvBnRelu3d(1, 32), ConvBnRelu3d(32, 32),
            nn.MaxPool3d(kernel_size=(2, 1, 1)),
        )
        self.enc3d_2 = nn.Sequential(
            ConvBnRelu3d(32, 64), ConvBnRelu3d(64, 64),
            nn.MaxPool3d(kernel_size=(2, 1, 1)),
        )
        self.enc3d_3 = nn.Sequential(
            ConvBnRelu3d(64, 128),
            nn.AdaptiveAvgPool3d((1, None, None)),
        )
        self.dec = nn.Sequential(
            ConvBnRelu2d(128, 64),
            ConvBnRelu2d(64, 32),
            ConvBnRelu2d(32, 16),
        )
        self.head = nn.Conv2d(16, 1, kernel_size=1)

    def forward(self, x):
        x = self.enc3d_3(self.enc3d_2(self.enc3d_1(x.unsqueeze(1)))).squeeze(2)
        return self.head(self.dec(x))


model = VesuviusNet().to(DEVICE)
print(f'Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

## 5 — Loss & metrics

In [ ]:
def dice_loss(pred, target, smooth=1.0):
    pred = torch.sigmoid(pred).view(-1)
    target = target.view(-1)
    return 1 - (2 * (pred * target).sum() + smooth) / (pred.sum() + target.sum() + smooth)

def bce_dice_loss(pred, target):
    return 0.5 * F.binary_cross_entropy_with_logits(pred, target) + 0.5 * dice_loss(pred, target)

def fbeta_score(pred_mask, true_mask, beta=0.5, eps=1e-6):
    tp = (pred_mask * true_mask).sum()
    fp = (pred_mask * (1 - true_mask)).sum()
    fn = ((1 - pred_mask) * true_mask).sum()
    p  = tp / (tp + fp + eps)
    r  = tp / (tp + fn + eps)
    return (1 + beta**2) * p * r / (beta**2 * p + r + eps)

## 6 — Build dataloaders (leave-one-out on fragments)

In [ ]:
fragment_dirs = sorted(CFG['train_dir'].glob('fragment*'))
print('Fragments found:', [f.name for f in fragment_dirs])

train_patches, val_patches = [], []
for frag_dir in fragment_dirs:
    print(f'Loading {frag_dir.name}...', end=' ')
    vol     = load_surface_volume(frag_dir, CFG['z_start'], CFG['z_end'])
    mask    = load_mask(frag_dir)
    labels  = load_labels(frag_dir)
    patches = extract_patches(vol, mask, labels, CFG['patch_size'], CFG['stride'])
    print(f'{len(patches)} patches')
    (val_patches if frag_dir.name == CFG['val_fragment'] else train_patches).extend(patches)

print(f'Train: {len(train_patches)} patches | Val: {len(val_patches)} patches')

train_dl = DataLoader(VesuviusDataset(train_patches, augment=True),
                      batch_size=CFG['batch_size'], shuffle=True, num_workers=2, pin_memory=True)
val_dl   = DataLoader(VesuviusDataset(val_patches),
                      batch_size=CFG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)

## 7 — Training loop

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG['num_epochs'], eta_min=1e-6)
scaler    = torch.cuda.amp.GradScaler(enabled=DEVICE.type == 'cuda')

best_fbeta = 0.0
history    = {'train_loss': [], 'val_loss': [], 'val_fbeta': []}


def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for vols, lbls in tqdm(loader, desc='train' if train else 'val  ', leave=False):
            vols, lbls = vols.to(DEVICE), lbls.to(DEVICE)
            if train: optimizer.zero_grad()
            with torch.cuda.amp.autocast(enabled=DEVICE.type == 'cuda'):
                logits = model(vols)
                loss   = bce_dice_loss(logits, lbls)
            if train:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            total_loss += loss.item()
            if not train:
                all_preds.append(torch.sigmoid(logits).cpu().numpy())
                all_labels.append(lbls.cpu().numpy())
    if train:
        return total_loss / len(loader), None
    preds  = (np.concatenate(all_preds).flatten()  > CFG['threshold']).astype(np.float32)
    labels = np.concatenate(all_labels).flatten().astype(np.float32)
    fb = fbeta_score(torch.tensor(preds), torch.tensor(labels)).item()
    return total_loss / len(loader), fb


for epoch in range(1, CFG['num_epochs'] + 1):
    tr_loss, _       = run_epoch(train_dl, train=True)
    vl_loss, vl_fb   = run_epoch(val_dl,   train=False)
    scheduler.step()
    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['val_fbeta'].append(vl_fb)
    star = ' ★' if vl_fb > best_fbeta else ''
    print(f'Epoch {epoch:02d}/{CFG["num_epochs"]}  train={tr_loss:.4f}  val={vl_loss:.4f}  F0.5={vl_fb:.4f}{star}')
    if vl_fb > best_fbeta:
        best_fbeta = vl_fb
        torch.save(model.state_dict(), CFG['model_dir'] / 'best_model.pth')

print(f'Best F0.5: {best_fbeta:.4f}')

## 8 — Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history['train_loss'], label='train')
axes[0].plot(history['val_loss'],   label='val')
axes[0].set_title('BCE + Dice loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()
axes[1].plot(history['val_fbeta'], color='green')
axes[1].set_title('Validation F0.5'); axes[1].set_xlabel('Epoch')
plt.tight_layout()
plt.savefig('training_curves.png', dpi=120)
plt.show()

## 9 — Inference on PHercParis4 scroll segments

Sliding window over each segment's full surface volume, overlap-averaged.

Segments in `data/scroll/` were downloaded by `download_segments.py` (~15 GB total, 7 segments, all 65 layers).

In [ ]:
model.load_state_dict(torch.load(CFG['model_dir'] / 'best_model.pth', map_location=DEVICE))
model.eval()


@torch.no_grad()
def infer_segment(vol: np.ndarray, patch_size=224, stride=112, batch_size=16) -> np.ndarray:
    """Sliding-window inference → probability map (H, W) via overlap averaging."""
    _, H, W  = vol.shape
    pred_sum = np.zeros((H, W), np.float32)
    count    = np.zeros((H, W), np.float32)
    positions = [(y, x) for y in range(0, H - patch_size + 1, stride)
                         for x in range(0, W - patch_size + 1, stride)]

    for i in tqdm(range(0, len(positions), batch_size), desc='inference'):
        batch_pos  = positions[i:i+batch_size]
        batch_vols = np.stack([vol[:, y:y+patch_size, x:x+patch_size] for y, x in batch_pos])
        with torch.cuda.amp.autocast(enabled=DEVICE.type == 'cuda'):
            probs = torch.sigmoid(model(torch.from_numpy(batch_vols).float().to(DEVICE))).cpu().numpy()[:, 0]
        for prob, (y, x) in zip(probs, batch_pos):
            pred_sum[y:y+patch_size, x:x+patch_size] += prob
            count   [y:y+patch_size, x:x+patch_size] += 1

    return pred_sum / np.maximum(count, 1)

In [ ]:
from skimage import measure, morphology

scroll_dirs = [CFG['scroll_dir'] / seg_id for seg_id in CFG['scroll_segments']
               if (CFG['scroll_dir'] / seg_id).exists()]
print(f'Scroll segments ready: {len(scroll_dirs)}/{len(CFG["scroll_segments"])}')
if len(scroll_dirs) < len(CFG['scroll_segments']):
    print('  Run download_segments.py to fetch missing segments.')

for seg_dir in scroll_dirs:
    seg_id = seg_dir.name
    print(f'\nSegment {seg_id}')

    vol  = load_surface_volume(seg_dir, CFG['z_start'], CFG['z_end'])
    mask = load_mask(seg_dir)
    if mask is None:
        mask = np.ones(vol.shape[1:], np.uint8)

    prob_map = infer_segment(vol, CFG['patch_size'], CFG['stride']) * mask

    # Save probability map
    imageio.imwrite(str(CFG['pred_dir'] / f'{seg_id}_prob.png'),
                    (prob_map * 255).astype(np.uint8))

    # Binarise + post-process
    binary  = (prob_map > CFG['threshold']).astype(np.uint8)
    labeled = measure.label(binary)
    for region in measure.regionprops(labeled):
        if region.area < CFG['min_cc_size']:
            binary[labeled == region.label] = 0
    binary = morphology.binary_closing(binary, morphology.disk(2)).astype(np.uint8)

    imageio.imwrite(str(CFG['pred_dir'] / f'{seg_id}_ink.png'), binary * 255)
    print(f'  → predictions/{seg_id}_prob.png  predictions/{seg_id}_ink.png')

    del vol; gc.collect()

## 10 — Visualise predictions

In [ ]:
def visualise_segment(seg_id: str, z_mid: int = 32):
    seg_dir   = CFG['scroll_dir'] / seg_id
    prob_img  = np.array(Image.open(CFG['pred_dir'] / f'{seg_id}_prob.png'))
    ink_img   = np.array(Image.open(CFG['pred_dir'] / f'{seg_id}_ink.png'))
    mid_path  = sorted((seg_dir / 'surface_volume').glob('*.tif'))[z_mid]
    mid_slice = tifffile.imread(str(mid_path)).astype(np.float32)
    mid_slice = ((mid_slice - mid_slice.min()) / (mid_slice.max() - mid_slice.min() + 1e-8) * 255).astype(np.uint8)

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(mid_slice, cmap='gray');  axes[0].set_title(f'Surface vol slice {z_mid}')
    axes[1].imshow(prob_img,  cmap='hot');   axes[1].set_title('Ink probability')
    axes[2].imshow(ink_img,   cmap='gray');  axes[2].set_title('Binarised ink mask')
    for ax in axes: ax.axis('off')
    plt.suptitle(f'Segment {seg_id}', fontsize=13)
    plt.tight_layout()
    plt.savefig(f'predictions/{seg_id}_vis.png', dpi=120)
    plt.show()


# Visualise Grand Prize segment first
visualise_segment('20230827161847')

## 11 — (Optional) LLM deciphering via Claude API

In [ ]:
# pip install anthropic  |  set ANTHROPIC_API_KEY env var before running

import base64
import anthropic


def extract_text_lines(ink_mask: np.ndarray, min_height=20, padding=10):
    """Horizontal projection profile to detect text-line bounding rows."""
    proj, in_line, lines = ink_mask.sum(axis=1), False, []
    for i, v in enumerate(proj):
        if v > 0 and not in_line:
            start, in_line = i, True
        elif v == 0 and in_line:
            in_line = False
            if i - start >= min_height:
                lines.append((max(0, start - padding), min(len(proj), i + padding)))
    return lines


def decipher_patch(patch_gray: np.ndarray, client: anthropic.Anthropic) -> str:
    _, buf = cv2.imencode('.png', patch_gray)
    b64 = base64.standard_b64encode(buf).decode()
    msg = client.messages.create(
        model='claude-opus-4-7',
        max_tokens=256,
        messages=[{'role': 'user', 'content': [
            {'type': 'image', 'source': {'type': 'base64', 'media_type': 'image/png', 'data': b64}},
            {'type': 'text', 'text':
             'This is a binarised ink detection patch from a Herculaneum papyrus (X-ray CT). '
             'White pixels = ink. Transcribe any visible ancient Greek characters, '
             'or respond [illegible] if nothing is readable.'},
        ]}],
    )
    return msg.content[0].text


# client   = anthropic.Anthropic()
# seg_id   = '20230827161847'
# ink_mask = np.array(Image.open(CFG['pred_dir'] / f'{seg_id}_ink.png')) > 127
# for i, (y0, y1) in enumerate(extract_text_lines(ink_mask.astype(np.uint8))[:5]):
#     patch = (ink_mask[y0:y1] * 255).astype(np.uint8)
#     print(f'Line {i}: {decipher_patch(patch, client)}')

## 12 — (Optional) Use pretrained Grand Prize weights instead of training

In [ ]:
# Skip training entirely — use the community pretrained UNet3D ensemble (Kaggle top-1):
# !pip install kaggle
# !kaggle datasets download -d ryches/unet3d -p models/
# !unzip models/unet3d.zip -d models/

# Or clone the Grand Prize repo:
# !git clone https://github.com/younader/Vesuvius-Grandprize-Winner grandprize
print('Pretrained weight commands are commented above.')